<a href="https://colab.research.google.com/github/LuciaMellini/AMD_project/blob/main/findingSimilarItems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding similar items

We download the Letterboxd dataset from Kaggle, using a token.

In [76]:
import os
import json
import pandas as pd
import pip
import string
import re
import numpy as np

os.environ['KAGGLE_USERNAME'] = "xxx"
os.environ['KAGGLE_KEY'] = "xxx"

In [77]:
#! kaggle datasets download -d gsimonx37/letterboxd

We only consider a subset of the files contained in the `letterboxd` dataset, namely the data regarding the movie names and ids, their actors, crews, genres and themes.

In [78]:
# import zipfile
# from multiprocessing import Pool

DATA_DIR = "./letterboxd"
# members_to_extract = ['actors', 'crew', 'genres', 'movies', 'themes']
# with zipfile.ZipFile(DATA_DIR + ".zip","r") as zip_ref:
#     for file_name in members_to_extract:
#         zip_ref.extract(file_name + '.csv', DATA_DIR)
!tar xf ./drive/MyDrive/letterboxd.tar.gz

We then prepare the entry point for the Spark functionalities that will we use from now on.

In [79]:
!apt-get install openjdk-21-jdk-headless -qq > /dev/null
#!wget https://downloads.apache.org/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz
!tar xf ./drive/MyDrive/spark-3.5.3-bin-hadoop3.tgz
!pip install -q findspark

In [80]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["SPARK_HOME"] = "./drive/MyDrive/spark-3.5.3-bin-hadoop3"

import findspark
findspark.init("spark-3.5.3-bin-hadoop3")
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.executor.memory", "2g") \
    .appName("ColabSpark") \
    .getOrCreate()

sc = spark.sparkContext

We begin by getting the input files into RDD form. We bring all the strings  to lower case to facilitate later manipulation.

In [81]:
import csv
from io import StringIO
SUB_FUNC = lambda s: re.sub(r'(\d+),[\s\t]+([a-zA-Z])', r'\1,\2', s)      # some actor names are written with a heading space, we exploit that the preceeding attribute is numerical (the movie id)

def csv_to_rdd(filename):
    raw = sc.textFile(filename).map(lambda s: s.lower())
    def parse_csv(line):
        reader = csv.reader(StringIO(line))
        return next(reader)
    return raw.map(parse_csv)
    # raw = sc.textFile(filename)
    # result = (raw
    #           .map(lambda s: s.replace('"', ''))
    #           .map(lambda s: re.sub(r'\s+', ' ', s))
    #           .map(lambda s: s.lower())
    #           .map(SUB_FUNC if filename.startswith('actors') else lambda s: s)
    #           .map(lambda r: re.split(r',(?! )', r)))  #split only on commas that are followed by a character to avoid splitting sentences e.g. in movie descriptions
    # return result

def get_column_names(rdd):
    column_names = rdd.filter(lambda r: r[0]=='id').collect()[0][1:]
    return column_names

def prepare_data(filename):
    rdd = csv_to_rdd(filename).cache()
    column_names = get_column_names(rdd)
    result = (rdd
            .filter(lambda r: r[0]!='id')
            .map(lambda r: (int(r[0]), dict(zip(column_names, r[1:])))))
    return result


In [82]:
members_to_extract = ['actors', 'crew', 'genres', 'movies', 'themes']
letterboxd_RDDs={}

for member in members_to_extract:
    letterboxd_RDDs[member] = prepare_data(os.path.join(DATA_DIR, f"{member}.csv"))

A glimpse at the structure of the rows in the RDDs of each member.

In [83]:
for member in members_to_extract:
    print(f"Row for {member:}:\t {letterboxd_RDDs[member].first()}")

Row for actors:	 (1000001, {'name': 'margot robbie', 'role': 'barbie'})
Row for crew:	 (1000001, {'role': 'director', 'name': 'greta gerwig'})
Row for genres:	 (1000001, {'genre': 'comedy'})
Row for movies:	 (1000001, {'name': 'barbie', 'date': '2023', 'tagline': "she's everything. he's just ken.", 'description': 'barbie and ken are having the time of their lives in the colorful and seemingly perfect world of barbie land. however, when they get a chance to go to the real world, they soon discover the joys and perils of living among humans.', 'minute': '114', 'rating': '3.86'})
Row for themes:	 (1000001, {'theme': 'humanity and the world around us'})


Below we have prepared a function to extract a sample of the data, based on the ids in the datasets. The maximum size of the sample is $125641$.




In [84]:
ID_MIN = 1000000

def get_sample(rdd, size):
    """ Extract a sample of records from the RDD based on a specified size

    Args:
        rdd (pyspark.RDD): The input RDD containing records, where each record's first element is expected
                           to be an ID as a string.
        size (int): The desired number of records to sample. The function filters records with IDs less
                    than or equal to 1,000,000 plus the specified size.

    Returns:
        pyspark.RDD: An RDD containing the filtered sample of records.
    """
    return rdd.filter(lambda r: r[0]<= ID_MIN+size)

In [85]:
sample_size = 100
sample_size_bc = sc.broadcast(sample_size)
letterboxd_RDDs_sample = {}
for member in members_to_extract:
    letterboxd_RDDs[member] = get_sample(letterboxd_RDDs[member], sample_size_bc.value)

For each member the available attributes are the following:

| **Member**    | **Attributes**                             |
|--------------|-----------------------------------------|
| **actor**    | name, role                          |
| **crew**     | role, name                          |
| **genres**   | genres                               |
| **movies**   | name, date, tagline, description, minute, rating |
| **themes**   | themes                               |
</br>

For this project we would like to focus on the following features:
<a name="table1"></a>

| **Member**    | **Attributes**                             |
|--------------|-----------------------------------------|
| **actor**    | names of the first 6 actors for a given movie                      |
| **crew**     | name(s) of the director of each movie                        |
| **genres**   | genres                               |
| **movies**   | name, date, minute, rating |
| **themes**   | themes                               |


We set up some primitives to manipulate the values in the RDDs' rows, that are of type `dict`.

In [86]:
from functools import reduce, partial
reduce_dicts = lambda x, y: {
        key: (x[key] + [y[key]] if isinstance(x[key], list) else [x[key], y[key]])
        for key in x.keys()
    }

remove_dict_item = lambda d, key: (d.pop(key), d)[1] if key in d else d

rename_key_in_dict = lambda d, old_key, new_key: remove_dict_item({**d, new_key: d.pop(old_key)} if old_key in d else d, old_key)

update_dict_value = lambda d, key, new_value: {**d, **{key: new_value}} if key in d else d

apply_dict_operations = lambda d, ops: reduce(lambda acc, op: op(acc), ops, d)

For the *crew* member we only keep the directors, and only their name.

In [87]:
operations = [
    partial(rename_key_in_dict, old_key='name', new_key='director'),
    partial(remove_dict_item, key='role')
]

letterboxd_RDDs['crew'] = (letterboxd_RDDs['crew']
                            .filter(lambda r: r[1]['role']=='director')
                            .map(lambda r: (r[0], apply_dict_operations(r[1], operations))))

Some examples of polished *crew* tuples.

In [88]:
letterboxd_RDDs['crew'].take(5)

[(1000001, {'director': 'greta gerwig'}),
 (1000002, {'director': 'bong joon-ho'}),
 (1000003, {'director': 'daniel scheinert'}),
 (1000003, {'director': 'daniel kwan'}),
 (1000004, {'director': 'david fincher'})]

For each category we account for movies having multiple values for a given attribute. In general given rows $$(id_1, \{k_{a}: v_{a_1}, k_{b}: v_{b_1}, \dots, k_{i}: v_{i_1}\dots\})\\(id_1, \{k_{a}: v_{a_2}, k_{b}: v_{b_2}, \dots, k_{i}: v_{i_2}\dots\})$$ the following reduction produces a row  $$(id_1, \{k_{a}: [v_{a_1}, v_{a_2}], k_{b}: [v_{b_1}, v_{b_2}], \dots, k_{i}: [v_{i_1}, v_{i_2}]\dots\})$$

In [89]:
for member in members_to_extract:
    letterboxd_RDDs[member] = letterboxd_RDDs[member].reduceByKey(lambda a, b: reduce_dicts(a, b))

Concerning the information in the *actor* member, we keep only the names of the  most relevant `n_actors` (actors) in each movie.

In [90]:
n_actors = 6

operations_actors = [
    partial(remove_dict_item, key='role'),
    partial(rename_key_in_dict, old_key='name', new_key='actors'),
    lambda d: update_dict_value(d, 'actors', d['actors'][:n_actors]),
    lambda d: {**d, **{f'actor{i+1}': actor for i, actor in enumerate(d['actors'])}},
    partial(remove_dict_item, key='actors')
]

letterboxd_RDDs['actors'] = (letterboxd_RDDs['actors']
                            .map(lambda r: (r[0], apply_dict_operations(r[1], operations_actors))))

For example,

In [91]:
letterboxd_RDDs['actors'].first()

(1000002,
 {'actor1': 'song kang-ho',
  'actor2': 'lee sun-kyun',
  'actor3': 'cho yeo-jeong',
  'actor4': 'choi woo-shik',
  'actor5': 'park so-dam',
  'actor6': 'lee jung-eun'})

For each *movie* we only store the attributes listed in the <a href="#table1">table above</a>.

In [92]:
operations_movies = [
    partial(remove_dict_item, key='tagline'),
    partial(remove_dict_item, key='description')
]

letterboxd_RDDs['movies'] = (letterboxd_RDDs['movies']
                            .map(lambda r: (r[0], apply_dict_operations(r[1], operations_movies))))

Now that we have polished each member in the dataset, we collect all the features for a given film in a single dictionary. So, a generic row of `movies_RDD` has the following format:
<p align=center><i>(id, {category: value(s)})</i></p>

In [93]:
movies_RDD = letterboxd_RDDs[members_to_extract[0]]
for member in members_to_extract[1:]:
    movies_RDD = movies_RDD.union(letterboxd_RDDs[member])
movies_RDD = movies_RDD.reduceByKey(lambda a, b: {**a, **b})

For example,

In [94]:
id, value = movies_RDD.first()
print("id:\t {}\nfeatures:\t {}".format(id,value['name']))

id:	 1000086
features:	 us


## Data pre-processing

To preserve the independent role of each attribute we have decided to measure their similarity using cosine distance. This entails translating all data regarding a movie into a vector with components in $\mathbb{R}$.

Below we list the data types of the various attributes.

| **Attribute**    | **Datatype**                       |
|--------------|-----------------------------------------|
| **actor**    | string         |
| **director**     | string                        |
| **genre**   | string                             |
| **theme**   | string                             |
| **name**   | string |
| **date**   | numerical |
| **minute**   | numerical |
| **rating**   | numerical                             |

It is evident that the textual attributes have to be transformed into values to be able to work in a Euclidean space. The following paragraphs are dedicated to these transformations. We refer to the report for a discussion regarding the chosen methods.


### String pre-processing

In [95]:
categories_all = [f'actor{i+1}' for i in range(n_actors)]+['director', 'genre', 'name', 'theme', 'date', 'minute', 'rating']

In [96]:
apply_to_list_or_value = lambda func, value: [func(v) for v in value] if isinstance(value, list) else func(value)

In [97]:
movies_processed_RDD = movies_RDD

#### Natural language processing

To distill the semantics of the movie's theme we apply the following natural language processing steps:
* remove stop words
* replace the words with their lemmatized version

In [98]:
import spacy
! python -m spacy download en_core_web_md -q

nlp = spacy.load("en_core_web_md")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 11.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [99]:
remove_punctuation = lambda x: re.sub(r'[^\w\s]','',x)
remove_multiple_spaces = lambda x: re.sub(r'\s+',' ',x)
remove_stop_words_func = lambda x: " ".join([token.text for token in nlp(x) if not token.is_stop])
lemmatize_func = lambda x: " ".join([token.lemma_ for token in nlp(x)])

chain_functions = lambda *funcs: lambda x: reduce(lambda acc, f: f(acc), funcs, x)
nlp_processing = chain_functions(remove_punctuation, remove_multiple_spaces, remove_stop_words_func, lemmatize_func)

categories_nlp = ['theme']

operations_nlp = [lambda d, cat=cat: update_dict_value(d, cat, apply_to_list_or_value(nlp_processing, d[cat])) for cat in categories_nlp]
movies_processed_RDD = movies_processed_RDD.map(lambda r: (r[0], apply_dict_operations(r[1], operations_nlp)))

#### Embedding strings

We use the spaCy text vectorizations of the attributes:
* name
* genre
* theme

In [100]:
embedding_func = lambda x: nlp(x).vector

categories_vec = ['name', 'genre', 'theme']

operations_vec = [lambda d, cat=cat: update_dict_value(d, cat, apply_to_list_or_value(embedding_func, d[cat])) for cat in categories_vec]
movies_processed_RDD = movies_processed_RDD.map(lambda r: (r[0], apply_dict_operations(r[1], operations_vec)))

For example a value for the *name* attribute will result as such:

In [101]:
id, value = movies_processed_RDD.first()
print("id:\t {}\nvalue:\t {}".format(id,value['name']))

id:	 1000040
value:	 [ 0.41696003  0.03877506  3.0641098   0.2637275   2.7442      1.524015
  4.186727   -1.7658577  -1.83165    -1.531274    1.194775   -2.46667
 -4.585125    0.17286623  2.5232759   0.2611375   3.0152502   0.48564753
 -0.07288504  0.741425   -2.37325     1.9866364   0.6204283   0.47031757
  2.3508      2.8903425  -0.509165    0.17997503  0.9963027   2.250425
  2.581465    0.65702003  1.1277997  -1.4702251  -1.6700313  -5.0861745
  1.1908917   0.328825   -3.3855      2.90865     3.1368425  -1.9526049
 -3.9421      3.0712497  -1.10318     0.48152256 -2.479875   -2.9893498
  3.7742848  -2.5631025   0.10997748  0.5116749   4.05915    -3.08589
 -4.486105    2.3370523  -1.078525   -0.12675256  1.125175    2.767375
 -0.39098048  2.0029726  -0.9843227  -0.23234987  1.0883499   2.9676101
 -2.1621125  -2.6365252   4.364725    0.15456748 -0.28221744  3.1322448
 -2.142775   -0.69993    -1.268355    0.64260006 -0.6703117   0.4209475
  1.4957888   1.9749274  -0.86789995 -3.18295   

Since for each movie there are multiple genres and themes, for these attributes the dictionary contains a list of numpy arrays.

#### Hashing strings

We hash the strings of the features:
* actors
* director

In [102]:
import math
import hashlib

def hash_object(byte_obj, hash_bucket_size, salt=bytes(0)):
    """ Hash an object into a bucket of values [0,hash_bucket_size-1] on the basis of the seed

        Args:
            byte_obj (byte array): an object in byte format
            salt (byte array): salt for the hash function
            hash_bucket_size (int): the size of the bucket to which the objects get hashed to

        Returns:
            int: hashed object
        """
    m=hashlib.shake_256()
    m.update(salt)
    m.update(byte_obj)
    required_bytes = math.ceil(hash_bucket_size / 8)
    hash_output = m.digest(required_bytes)
    hashed_value = int.from_bytes(hash_output, 'little')
    return hashed_value % hash_bucket_size

In [103]:
hash_func = lambda v: hash_object(bytes(v,'utf-8'),2**30)

categories_hash = [f"actor{i+1}" for i in range(n_actors)]+['director']

operations_hash = [lambda d, cat=cat: update_dict_value(d, cat, apply_to_list_or_value(hash_func, d[cat])) for cat in categories_hash]
movies_processed_RDD = movies_processed_RDD.map(lambda r: (r[0], apply_dict_operations(r[1], operations_hash)))

For example a value for the *director* attribute will result as such:

In [104]:
id, value = movies_processed_RDD.first()
print("id:\t {}\nvalue:\t {}".format(id,value['director']))

id:	 1000040
value:	 [532324926, 687849396]


And if a movie has multiple directors the value is a list of hashes.

In [105]:
id, value = movies_processed_RDD.filter(lambda r: isinstance(r[1]['director'], list)).first()
print("id:\t {}\nvalue:\t {}".format(id,value['director']))

id:	 1000040
value:	 [532324926, 687849396]


### Vector preparation

Now that we have prepared all categories in a targeted way, we can proceed by building the vectors of the movies.

#### Reduce attributes with multiple values

We simply average the values or arrays for attributes that have multiple values. e.g directors, themes, genres.

In [106]:
# cannot assume that the value is a list since e.g. "director" attribute could have a singular value
def average_list_values(v):
    if isinstance(v, list):
        if isinstance(v[0],np.ndarray):
            return np.mean(np.array(v), axis=0)
        return np.mean(v)
    return v

In [131]:
operations_average = [lambda d, cat=cat: update_dict_value(d, cat, average_list_values(d[cat])) for cat in categories_all]
vectors_RDD = movies_processed_RDD.map(lambda r: (r[0], apply_dict_operations(r[1], operations_average)))

In [108]:
def dictionary_to_array_of_values(d):
    return np.concatenate([v if isinstance(v, np.ndarray) else np.array([v]) for key, v in sorted(d.items())])

In [132]:
vectors_RDD = (vectors_RDD.map(lambda r: (r[0], dictionary_to_array_of_values(r[1])))
                .map(lambda r: (r[0], np.vectorize(lambda x: float(x))(r[1])))).cache()

In [110]:
vectors_RDD.first()

(1000040,
 array([ 2.39007945e+08,  2.84744443e+08,  7.73779779e+08,  7.95950406e+08,
         6.84320722e+08,  3.30743590e+07,  2.01800000e+03,  6.10087161e+08,
        -1.27584340e+00, -1.12384670e+00, -1.48066650e-01, -1.36658330e+00,
         3.14748300e-01,  2.41065240e+00,  2.42325660e+00,  2.63320000e+00,
        -1.22485170e+00, -2.17265650e+00,  4.39576670e+00,  1.93015660e+00,
        -4.35713340e+00,  2.07823280e-01,  1.77226010e+00,  2.51986340e+00,
         6.54741670e+00,  4.40768340e+00, -4.72057870e+00,  1.17319980e-01,
         2.48035340e+00,  1.08609830e+00, -3.63378330e+00,  2.85295160e-01,
        -7.47674940e-01, -1.68619330e+00, -1.12043330e+00, -3.62791660e+00,
        -2.01035120e+00,  3.02322650e+00,  2.67460820e+00,  3.33890820e+00,
         9.44431600e-01,  1.22153320e-01, -1.28256640e+00, -1.30934000e+00,
         2.54035000e+00,  3.03749920e-01, -1.75716340e+00, -1.69071670e+00,
        -1.92451310e-01, -1.24338330e+00, -4.58866720e-01,  1.20546660e+00,
  

For each movie we have a vector of the following dimension:

In [111]:
dim_vectors = len(vectors_RDD.first()[1])
print(f"Each vector has {dim_vectors} dimensions")

Each vector has 910 dimensions


#### Standardization

We standardize the vector's components such that each feature has a sample mean of $0$ and a sample standard deviation of $1$.

In [121]:
def get_mean_per_list_index(rdd):
    rdd = (rdd
           .flatMap(lambda r: [(i,(item, 1)) for i,item in enumerate(r[1])])
           .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) )
    rdd_mean = rdd.map(lambda r: (r[0], r[1][0]/r[1][1]))
    rdd_mean.coalesce(1)
    return dict(rdd_mean.collect())

def get_ssd_per_list_index(rdd, mean_dict):
    rdd_sum_sq_diff_count = (rdd
            .flatMap(lambda r: [(i,((item - mean_dict[i])**2, 1)) for i,item in enumerate(r[1])])
            .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])))
    rdd_ssd = rdd_sum_sq_diff_count.map(lambda r: (r[0], math.sqrt(r[1][0]/r[1][1]))).cache()
    rdd_ssd.coalesce(1)
    return dict(rdd_ssd.collect())

In [125]:
mean_dict = get_mean_per_list_index(vectors_RDD)
mean_dict_bc = sc.broadcast(mean_dict)
ssd_dict = get_ssd_per_list_index(vectors_RDD, mean_dict_bc.value)
ssd_dict_bc = sc.broadcast(ssd_dict)

In [133]:
vectors_stand_RDD = vectors_RDD.map(lambda r:  (r[0], (r[1] - np.array([mean_dict_bc.value[i] for i in range(len(r[1]))])) / np.array([ssd_dict_bc.value[i] for i in range(len(r[1]))])))

In [134]:
vectors_stand_RDD.first()

(1000040,
 array([-9.90228574e-01, -1.01093198e+00,  7.42967086e-01,  9.03670610e-01,
         5.34699161e-01, -1.57967839e+00,  6.17528974e-01,  2.48989388e-01,
        -1.03954114e+00, -3.72945344e-01,  8.53637800e-01,  1.07774908e+00,
         6.66376850e-01,  1.48191330e+00, -1.39767069e+00,  6.21471499e-01,
        -1.52977049e+00,  6.31934340e-01,  1.19866254e+00,  1.30124933e+00,
        -1.59675299e+00,  2.86795131e-01,  8.25789938e-01,  1.52622978e+00,
         1.10706920e+00,  9.98010439e-01, -1.61004871e+00, -1.20759346e+00,
         1.10568130e+00, -8.19324562e-01, -1.71675414e+00,  9.65957543e-01,
        -2.90696566e-01, -3.89952032e-01,  3.21642845e-01, -1.74067254e+00,
        -7.69784140e-01,  1.33999793e+00,  1.54226197e+00,  4.07987520e-01,
        -4.84551837e-01, -6.44568720e-02,  4.53147035e-01, -1.61704883e+00,
         1.66204023e+00,  1.00423029e+00,  7.05301013e-01,  2.81448622e-01,
         4.75000229e-01, -8.17987557e-01, -1.37148302e+00,  1.43335638e+00,
  

#### Principal-Component Analysis (PCA)

We reduce the amount of features for each vector to avoid suffering from the curse of dimensionality when evaluating their similarity.

We begin by building the build the covariance matrix for the data, and we compute its eigenvalues and eigenvectors.

In [139]:
def PCA(vectors_rdd):
    """
    Perform Principal Component Analysis (PCA) on an RDD of vectors.

    Args:
        vectors_rdd (pyspark.RDD): An RDD where each element is a tuple with an identifier, and the second part is a vector of numerical features.

    Returns:
        tuple: A tuple containing:
            - sorted_eigenvalues (numpy.ndarray): The eigenvalues sorted in non increasing order.
            - sorted_eigenvectors (numpy.ndarray): The eigenvectors corresponding to the sorted eigenvalues.
    """
    n_vectors = vectors_rdd.count()
    cov_matrix = (vectors_rdd.map(lambda r: (r[0], np.outer(r[1], r[1])))
                .reduce(lambda a, b: (1, a[1] + b[1])))[1] / n_vectors

    eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

    # sort eigenvalues and eigenvectors in non increasing order
    sorted_indices = np.argsort(eigenvalues)[::-1]
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices]
    return (sorted_eigenvalues, sorted_eigenvectors)

In [140]:
sorted_eigenvalues, sorted_eigenvectors = PCA(vectors_stand_RDD)

We only retain the components such that their total cumulative explained variance is at least $95\%$.

In [142]:
def k_principal_components(sorted_eigenvalues, sorted_eigenvectors, t_PCA):
    """
    Select the top-k principal components based on the cumulative explained variance threshold.

    Args:
        sorted_eigenvalues (numpy.ndarray): The eigenvalues sorted in non increasing order.
        sorted_eigenvectors (numpy.ndarray): The eigenvectors corresponding to the sorted eigenvalues.
        t_PCA (float): The cumulative explained variance threshold (a value between 0 and 1).

    Returns:
        numpy.ndarray: The top-k eigenvectors that explain at least the specified threshold of the variance.
    """
    explained_variance = sorted_eigenvalues / np.sum(sorted_eigenvalues)
    cumulative_explained_variance = np.cumsum(explained_variance)
    k = np.argmax(cumulative_explained_variance >= t_PCA) + 1
    top_k_eigenvectors = sorted_eigenvectors[:, :k]
    return top_k_eigenvectors

In [143]:
t_PCA = 0.95
top_k_eigenvectors = k_principal_components(sorted_eigenvalues, sorted_eigenvectors, t_PCA)
k = top_k_eigenvectors.shape[1]

print(f"Number of components explaining at least {t_PCA*100}% of the variance: {k}")

Number of components explaining at least 95.0% of the variance: 53


In [144]:
top_k_eigenvectors_bc = sc.broadcast(top_k_eigenvectors)
vectors_reduced_RDD = vectors_stand_RDD.map(lambda r: (r[0], np.dot(top_k_eigenvectors_bc.value.T, r[1])))

An example of reduced vector:

In [146]:
id, vector = vectors_reduced_RDD.first()
print("movie id:\t {}\n  vector:\t {}".format(id,vector))

movie id:	 1000040
  vector:	 [25.45678573+0.j  0.12065804+0.j -2.63756383+0.j -1.42343519+0.j
  5.08507867+0.j -3.95027141+0.j  0.8219175 +0.j  3.2039606 +0.j
  0.13246169+0.j  2.9730251 +0.j  1.06559536+0.j  0.93789517+0.j
  0.23405023+0.j  1.91881201+0.j -0.54194437+0.j  1.44506389+0.j
  3.81372045+0.j -0.32028173+0.j  0.2941037 +0.j  2.31112813+0.j
 -0.93754253+0.j -1.14918825+0.j  1.27860049+0.j  1.08220835+0.j
 -1.56758995+0.j  2.54590296+0.j -1.45645182+0.j -1.49585892+0.j
 -1.70019421+0.j -2.57253945+0.j  0.9631534 +0.j  0.55967854+0.j
  1.05008809+0.j  0.68704178+0.j  0.48528741+0.j  0.99323773+0.j
 -2.20962485+0.j  1.5121077 +0.j  0.14521523+0.j -0.2523951 +0.j
  0.0701358 +0.j  1.65900701+0.j  2.18206555+0.j  1.075918  +0.j
 -0.83334205+0.j  4.14620218+0.j -0.08746862+0.j -1.99576198+0.j
 -2.37028838+0.j  1.23256378+0.j  0.05430903+0.j -0.67606319+0.j
  0.64904429+0.j]


## Locality Sensitive Hashing (LSH)

### Locality sentitive family for cosine distance

We build the locality sensitive family $\mathbf{F}$ as a set of randomly chosen vectors $\{v_{f\in\mathbf{F}}\}$. Given two vectors $x$ and $y$, they make a candidate pair of similar items if and only if the dot products $x\cdot v_f$ and $x \cdot v_f$ have the same sign. A family of functions $\mathbf{F}$ built as described is a locality-sensitive family for the cosine distance.

We will also refer to the random vectors in $\mathbf{F}$ as hash functions.

Since we will compute the dot product between each of the elements in the dataset and all of the hash functions in $\mathbf{F}$, we try to simplify the computation of the cosine distance between vectors. We do so by restricting the random choice of vectors to those having components $+1$ or $-1$. Hence the dot product of any vector $x$ with a vector in such a family $\mathbf{F}$ is given by its algebraic sum $x$'s components, where the signs depend on the components of the random vector.

In [149]:
def RDD_LS_hash_family_cosine_distance(signature_length, num_components):
    """
    Generate a locality sentitive family for cosine distance.

    Args:
        signature_length (int): The signature length for the items.
        num_components (int): he number of components for the random vectors.
    Returns:
        pyspark.RDD: An RDD of hash functions, where each hash function is a tuple with an index and a random vector in {-1,1}^k.
    """

    hash_funcs_RDD = (sc.parallelize([(i,1) for i in range(signature_length)])
                    .map(lambda r: (r[0],np.random.choice([-1, 1], num_components))))

    return hash_funcs_RDD

For each of the vectors in $\mathbf{F}$ (called `hash_funcs_RDD` in the code), we compute the dot products with each of the (reduced) vectors in the dataset.

In [153]:
def RDD_signatures(vectors_rdd, hash_funcs_rdd):
    """
    Generate locality-sensitive hashing (LSH) signatures for a set of vectors.

    Args:
        vectors_rdd (pyspark.RDD): An RDD where each element is a tuple. The first part is an identifier, and the second part is a vector of numerical features.
        signature_length (int): The signature length for the items.
    Returns:
        pyspark.RDD: An RDD where each element is a tuple containing an identifier and its corresponding LSH signature.
    """
    vectors_hash_RDD = (vectors_rdd.cartesian(hash_funcs_rdd)
                    .map(lambda r: ((r[0][0],r[1][0]), (r[0][1], r[1][1]))))
    signatures_RDD = (vectors_hash_RDD.map(lambda r: (r[0][0], (r[0][1], np.sign(np.dot(r[1][0], r[1][1])))))
                    .groupByKey().mapValues(list)
                    .mapValues(sorted)
                    .map(lambda r: (r[0], np.array([int(v[1]) for v in r[1]]))))                            # numpy uses complex numbers
    return signatures_RDD

In [154]:
signature_length=100
hash_funcs_RDD = RDD_LS_hash_family_cosine_distance(signature_length, k)
signatures_RDD = RDD_signatures(vectors_reduced_RDD, hash_funcs_RDD)

Let's give look at a possibile signature for a movie.

In [155]:
id, signature = signatures_RDD.first()
print(" movie id:\t {}\nsignature:\t {}".format(id,signature))

 movie id:	 1000040
signature:	 [ 1.+0.j  1.+0.j  1.+0.j -1.+0.j  1.+0.j -1.+0.j  1.+0.j -1.+0.j  1.+0.j
  1.+0.j -1.+0.j  1.+0.j  1.+0.j -1.+0.j  1.+0.j  1.+0.j  1.+0.j  1.+0.j
  1.+0.j  1.+0.j -1.+0.j  1.+0.j  1.+0.j  1.+0.j -1.+0.j -1.+0.j -1.+0.j
 -1.+0.j  1.+0.j  1.+0.j -1.+0.j -1.+0.j -1.+0.j  1.+0.j  1.+0.j  1.+0.j
 -1.+0.j -1.+0.j  1.+0.j  1.+0.j  1.+0.j -1.+0.j -1.+0.j  1.+0.j  1.+0.j
  1.+0.j  1.+0.j  1.+0.j -1.+0.j -1.+0.j  1.+0.j  1.+0.j  1.+0.j -1.+0.j
  1.+0.j -1.+0.j  1.+0.j -1.+0.j  1.+0.j  1.+0.j -1.+0.j  1.+0.j  1.+0.j
 -1.+0.j  1.+0.j  1.+0.j  1.+0.j  1.+0.j  1.+0.j  1.+0.j -1.+0.j  1.+0.j
  1.+0.j  1.+0.j -1.+0.j -1.+0.j -1.+0.j -1.+0.j  1.+0.j  1.+0.j -1.+0.j
 -1.+0.j -1.+0.j  1.+0.j  1.+0.j  1.+0.j -1.+0.j -1.+0.j  1.+0.j  1.+0.j
  1.+0.j -1.+0.j -1.+0.j  1.+0.j  1.+0.j  1.+0.j  1.+0.j  1.+0.j -1.+0.j
 -1.+0.j]


Now, `signatures_RDD` contains the so called *signature matrix*, that contains a signature for each movie in the dataset.

### Banding technique

It would be unthinkable to compare all possible pairs of movies to find similar ones among them. This would mean scanning all the rows in the signature matrix to compute the relative frequency between possible pairs of movies. So, we proceed by applying locality-sensitive hashing. In this approach we reduce the number of rows that determine the signature of a movie by hashing so called bands of rows. The rationale is that similar movies are more likely to be hashed in the same bucket, so we hope that dissimilar pairs end up in distinct buckets, and thus are never checked for similarity. Looking at the resulting signatures we consider as a candidate pair only those for which their cosine similarity exceeds a threshold $t$.

We begin by dividing the signature matrix into $b$ bands of $r$ rows each. The choice of $r$ and $b$ depends on the threshold $t$ on the cosine distance between pairs of movies. The value of the threshold $t$ is approximately the value of similarity at which the probability of becoming a candidate is $\frac{1}{2}$.
So, we keep into account the following relationships (for brevity $l$=`signature_length`):   

\begin{equation*}
    \begin{cases}
        t=\left(\frac{1}{b}\right)^\frac{1}{r} \\
        r\cdot b = l
    \end{cases}
\end{equation*}

The function `band_size` solves this system in $r$ by computing it's value through
\begin{equation*}
    r=-\frac{W(-l\ln(t))}{\ln(t)}
\end{equation*}

where $W$ is the Lambert $W$ function, used to solve equations in the form $we^{w}=z$ for $w$.

In [156]:
from scipy.special import lambertw
import math

def band_size(t, signature_length):
    """
    Compute the number of rows that form a band for the LSH technique.

    Args:
        t (int): The desired threshold in [0,1] on the similarity between pairs of items.
        signature_length (int): The signature length for the items.

    Returns:
        int: The ideal number of rows contained in the band.
    """
    return -math.ceil(lambertw(-signature_length*np.log(t)).real/np.log(t))

def r_b_choice(t,signature_length):
    """
    Choose the adeguate number of rows a band and the number of bands for the LSH technique.

    Args:
        t (int): The desired threshold in [0,1] on the similarity between pairs of items.
        signature_length (int): The signature length for the items.

    Returns:
        int: The ideal number of rows, and consequent number of bands on the basis of the signature length.
    """
    r=band_size(t,signature_length)
    b=math.ceil(signature_length/r)
    return (r,b)

In [157]:
t=0.8
t_bc = sc.broadcast(t)
r,b = r_b_choice(t,signature_length)
print("The chosen parameters are: \n r: {} \n b: {}".format(r,b))

The chosen parameters are: 
 r: 10 
 b: 10


Having chosen the parameters we proceed by subdividing the rows of the similarity matrix into bands.

In [158]:
def split_list(l,b):
    """ Split list into b lists of equal length

    Args:
        l (list): list of elements
        b (int): number of sublists

    Returns:
        list: list formed by b sublists all of the same length, except for the last one if len(l) is not a multiple of b
    """
    split_list = [(list(a)) for a in np.array_split(np.array(l), b)]
    return [split_list[i] for i in range(b)]

We proceed by hashing the rows in each of the $b$ bands for each vector. For each band we use a different bucket array, so that signatures with two equal vectors in separate bands are hashed differently. This is done by setting a hash function with a distinct seed for all bands.

In [168]:
def RDD_LSH(signatures_rdd, b, hash_bucket_size):
    """ Compute the hashed signatures with the LSH technique

    Args:
        signatures_rdd (RDD): RDD of (set key, signature for set)
        b (int): number of bands in which to split the signature matrix represented by signatures_rdd
        hash_bucket_size (int): the size of the bucket to which the signature portions in each band get hashed to

    Returns:

    """
    b_bc = sc.broadcast(b)
    hash_bucket_size_bc = sc.broadcast(hash_bucket_size)

    return (signatures_rdd
            .map(lambda r: (r[0],split_list(r[1],b_bc.value)))
            .map(lambda r: (r[0],[hash_object(bytes(str(t),'ascii'),hash_bucket_size_bc.value,bytes(i)) for i,t in enumerate(r[1])])))

Also, to avoid hashing distinct portions of a signature in the same bucket it is important to choose a great enough bucket. Here we have evaluted the number of tuples given by $\{-1,1\}^r$.

In [169]:
hash_bucket_size = 2**20
hashed_signatures_RDD = RDD_LSH(signatures_RDD, b, hash_bucket_size)

Let's give a look at the new compact representation of a movie.

In [170]:
id, signature = hashed_signatures_RDD.first()
print(" movie id:\t {}\nsignature:\t {}".format(id,signature))

 movie id:	 1000040
signature:	 [150552, 543657, 481192, 222588, 998547, 573197, 804321, 414841, 670305, 953024]


### Find similar items

Now we search for candidate pairs among the reviews. We consider as possible similar couples of reviews those that have cosine similarity at least $t$.

In [ ]:
def RDD_pairs(rdd):
    """ Put together all possible pairs of rows, without repetitions

    Args:
        rdd (RDD): RDD of (row key, information regarding row)

    Returns:
        RDD: RDD of ((r1,r2),(info1, info2)), with r1>r2 to avoid having duplicates
    """
    pairs_RDD = rdd.cartesian(rdd).filter(lambda r: r[0][0] > r[1][0])
    return pairs_RDD.map(lambda r: ((r[0][0], r[1][0]),(r[0][1], r[1][1])))



def RDD_candidate_pairs(rdd):
    """ Filter from pairs of rows whether they are candidate pairs

    Args:
        rdd (RDD): RDD of (row key, information regarding the row)

    Returns:
        RDD: RDD of ((r1,r2),(info1, info2)) such that exists i such that el_r1[i]==el_r2[i]
    """
    pairs_RDD = RDD_pairs(rdd)
    return pairs_RDD.filter(lambda r: any(x == y for x, y in zip(r[1][0], r[1][1])))

def cosine_distance(vec1,vec2):
    """
    Compute the cosine distance between two vectors.

    Args:
        vec1 (numpy.ndarray): The first vector.
        vec2 (numpy.ndarray): The second vector.

    Returns:
        float: The cosine distance between the two vectors, in radians.
    """
    cosine = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    return np.arccos(cosine)

def cosine_similarity(vec1,vec2):
    """
    Compute the cosine similarity between two vectors.

    Args:
        vec1 (numpy.ndarray): The first vector.
        vec2 (numpy.ndarray): The second vector.

    Returns:
        float: The cosine similarity between the two vectors.
    """

    return (math.pi-cosine_distance(vec1,vec2))/math.pi

def RDD_similar_items(rdd,t, similarity_function):
    """ Filter from pairs of rows whether they have similarity at least t

    Args:
        rdd (RDD): RDD of (row key, information regarding the row)
        t (int): desired threshold on the Jaccard similarity
        similarity_function (function): function to compute the similarity between two vectors

    Returns:
        RDD: RDD of ((k1,k2),1) if similarity_function(info1,info2)>=t
    """
    t_bc = sc.broadcast(t)
    return (rdd
            .filter(lambda r: similarity_function(r[1][0],r[1][1])>=t_bc.value)
            .map(lambda r: (r[0],similarity_function(r[1][0],r[1][1]))))

In [ ]:
candidate_pairs_RDD = RDD_candidate_pairs(hashed_signatures_RDD).cache()
similar_items_RDD = RDD_similar_items(candidate_pairs_RDD,t, cosine_similarity).cache()

## Experiments

In this paragraph we run some experiments with various combinations of the parameters. We have followed the outline described in Section 3.2.2 of the report.

In [ ]:
def data_pre_processing(data_rdd, t_PCA):
    ## we do no repeat steps that do not depend on the parameters for computational purposes
    # * polish data
    # * data embedding
    # * manage attributes with multiple values
    print("Standardizing vectors...")
    vectors_stand_RDD = RDD_standardize(data_rdd)
    print("Dimensionality reduction...")
    vectors_reduced_RDD, k = RDD_dimensionality_reduction_PCA(vectors_stand_RDD, t_PCA)
    return vectors_reduced_RDD, k

def find_similar_items(data_rdd, k, l, t, b, h):
    print("Creating locality sensitive family...")
    hash_funcs_RDD = RDD_LS_hash_family_cosine_distance(l, k).cache()
    print("Building signatures...")
    signatures_RDD = RDD_signatures(data_rdd, hash_funcs_RDD).cache()
    print("Applying banding technique...")
    hashed_signatures_RDD = RDD_LSH(signatures_RDD, b, h).cache()
    print("Finding candidate pairs...")
    candidate_pairs_RDD = RDD_candidate_pairs(hashed_signatures_RDD).cache()
    print("Filtering similar items...")
    similar_items_RDD = RDD_similar_items(candidate_pairs_RDD, t, cosine_similarity)
    return similar_items_RDD, candidate_pairs_RDD

def run_experiment(data_rdd, t_PCA, l, t, b, h):
    vectors_RDD, k = data_pre_processing(data_rdd, t_PCA)
    print(f"Number of components explaining at least {t_PCA*100}% of the variance: {k}")
    similar_items_RDD, candidate_pairs_RDD = find_similar_items(vectors_RDD, k, l, t, b, h)
    return similar_items_RDD, candidate_pairs_RDD

def save_results(similar_items_rdd, candidate_pairs_rdd, exp_name, *parameters):
    data = {
        "parameters": parameters,
        "similar_items": similar_items_rdd.collect(),
        "candidate_pairs": candidate_pairs_rdd.collect()
    }
    with open(f"results_{exp_name}.json", "w") as f:
        json.dump(data, f)

### Experiment 1

In [ ]:
parameters = (
    0.95,               #t_PCA
    100,                #l
    0.5,                #t
    2**20,              #h
)
parameters += (r_b_choice(parameters[2],parameters[1])[1],) #b

similar_items_RDD, candidate_pairs_RDD = run_experiment(vectors_RDD, *parameters)
print("Saving results...")
similar_items_RDD = similar_items_RDD.coalesce(1)
candidate_pairs_RDD = candidate_pairs_RDD.coalesce(1)
candidate_pairs_RDD.saveAsTextFile(f"candidate_pairs_exp_2.txt")
similar_items_RDD.saveAsTextFile(f"similar_items_exp_2.txt")
print(parameters)
with open(f"parameters_exp_2.json", "w") as f:
    json.dump(parameters, f)